In [ ]:
!pip install langchain langgraph langchain-community chromadb sqlite-utils

In [6]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. 상태 정의
class AgentState(TypedDict):
    query: str
    symptoms: str
    exercise_candidates: str
    result: str

# 2. LLM 초기화
llm = Ollama(model="exaone3.5:2.4b")

# 3. 에이전트(노드) 함수 정의

def extractor_agent(state: AgentState):
    extractor_prompt = PromptTemplate.from_template("질문에서 증상만 추출하세요: {query}")
    chain = extractor_prompt | llm
    symptoms = chain.invoke({"query": state["query"]})
    return {**state, "symptoms": symptoms.strip()}

def candidate_agent(state: AgentState):
    candidate_prompt = PromptTemplate.from_template("증상에 맞는 운동 3가지를 추천하세요: {symptoms}")
    chain = candidate_prompt | llm
    candidates = chain.invoke({"symptoms": state["symptoms"]})
    return {**state, "exercise_candidates": candidates.strip()}

def answer_agent(state: AgentState):
    answer_prompt = PromptTemplate.from_template("""
    증상: {symptoms}
    운동: {exercise_candidates}
    위 내용을 바탕으로 아래 형식에 맞춰 답변하세요.
    - 증상:
    - 추천 운동:
    - 운동 효과:
    """)
    chain = answer_prompt | llm
    answer = chain.invoke({
        "symptoms": state["symptoms"],
        "exercise_candidates": state["exercise_candidates"]
    })

    # 모든 특수문자 제거
    refined_answer = answer.replace("*", "").replace("#", "").strip()
    return {**state, "result": refined_answer}

# 4. LangGraph 정의 및 컴파일
graph = StateGraph(AgentState)

graph.add_node("extractor", extractor_agent)
graph.add_node("candidate", candidate_agent)
graph.add_node("answer", answer_agent)

graph.set_entry_point("extractor")
graph.add_edge("extractor", "candidate")
graph.add_edge("candidate", "answer")
graph.add_edge("answer", END)

app = graph.compile()

# 5. 실행
query = "체력이 안좋고, 살이 계속 찌는데 어떤 운동을 할까?"
result = app.invoke({"query": query})

print("[최종 응답]")
print(result["result"])

[최종 응답]
증상 분석 및 추천 운동 프로그램

증상:
- 체력 저하: 일상 활동에 대한 지속적인 피로감 증가
- 지속적인 체중 증가: 건강한 식단 관리에도 불구하고 체중 변화가 미미하거나 증가 추세

추천 운동:

1. 저강도 지구력 운동
   - 활동 예시:
     - 하루 걷기: 시작 시 하루 30분간의 가벼운 걷기로 시작하여, 점차 시간을 늘려 (예: 주 5일로 증가시키되, 일주일에 하루는 쉬는 날을 포함) 속도를 점진적으로 높이세요.
     - 수영: 주 2-3회, 각각 20-30분 정도의 수영을 실시합니다. 물의 부력 덕분에 관절에 덜 부담을 주며 전신 운동 효과를 얻을 수 있습니다.
   - 효과: 체력 향상에 도움이 되며, 부담 없이 참여할 수 있어 꾸준히 운동할 수 있는 동기 부여를 제공합니다.

2. 근력 훈련
   - 운동 세트 예시:
     - 풀 세트 루틴:
       - 월요일: 상체 - 스쿼트(12-15회), 푸쉬업(8-12회), 플랭크(30초) x 3세트
       - 화요일: 하체 - 스쿼트(12-15회), 런지(10-12회), 플랭크(30초) x 3세트
       - 수요일: 코어 강화 - 플랭크(30초 이상), 사이드 플랭크(각각 30초), 브리지(8회) x 3세트
       - 목요일: 팔과 어깨 - 푸쉬업(8-12회), 벤치 프레스(10-12회), 어깨 스트레칭 x 3세트
       - 금요일: 전체적인 균형 - 전신 풀 세트 루틴 (상체, 하체, 코어 병행) x 3세트
     - 활용 도구: StrengthLab 또는 MyFitnessPal 앱을 통해 개인 목표에 맞춘 트레이닝 계획을 설정하고 추적하세요.
   - 효과: 근력 강화는 기초 대사량 증가와 함께 지방 연소를 촉진하여 체중 관리에 효과적입니다. 체력 향상 또한 일상 활동의 효율성을 높입니다.

3. 유연성 및 균형 운동
   - 활동 예시:
     - 스트레칭 세션: 매일 10-15분 동안 전신 스트레칭을 실시합니다. 특히 허리, 가슴, 어

In [9]:
app.get_graph().draw_mermaid_png()

# 이미지 파일로 저장
with open("graph.png", "wb") as f:
    f.write(app.get_graph().draw_mermaid_png())